# 병합/분할

## 행병합
- merge: column기반 병합 (sql join과 유사)
- join: index기반 병합
- concat

In [1]:
import numpy as np
import pandas as pd
from altair.vegalite.v6 import display

In [2]:
df1 = pd.DataFrame({
    "ID": [1, 2, 3, 4],
    "Name": ['홍길동', '신사임당', '이순신', '논개']
})
df2 = pd.DataFrame({
    "ID": [1, 2, 3, 5],
    "Score": [85, 90, 78, 92]
})
display(df1)
display(df2)

TypeError: 'module' object is not callable

### merge
- merge: column기반 병합

In [ ]:
# inner : 두 df에 공통된 행만 병합
pd.merge(df1, df2, on='ID', how='inner')

In [ ]:
# left outer : 왼쪽 df의 모든 행 + 오른쪽 df에 공통된 행만 병합
pd.merge(df1, df2, on='ID', how='left')

In [ ]:
# right outer : 오른쪽 df의 모든 행 + 왼쪽 df에 공통된 행만 병합
pd.merge(df1, df2, on='ID', how='right')

In [ ]:
# outer : 오른쪽 df의 모든 행 + 왼쪽 df에 모든 행 병합
pd.merge(df1, df2, on='ID', how='outer')

### join
- join: index기반 병합

In [ ]:
customer_df = pd.DataFrame({
    'customer_id': [100, 200, 300],
    'name': ['홍길동', '신사임당', '윤봉길']
}).set_index('customer_id') # 일반컬럼을 index로 변환
display(customer_df)

order_df = pd.DataFrame({
    'customer_id': [100, 500],
    'order_id': ['o1', 'o2']
}).set_index('customer_id')
display(order_df)

In [ ]:
# left join (기본값)
# customer_df.join(order_df)
customer_df.join(order_df, how='left')

In [ ]:
# right join
customer_df.join(order_df, how='right')

In [ ]:
# inner join
customer_df.join(order_df, how='inner')

In [ ]:
# customer_df # customer_id가 인덱스로 지정된 상태
# customer_df.reset_index() # 0-based index 다시 사용 가능
customer_df.reset_index(drop=True) # 기존 인덱스 역할 컬럼을 제거하면서 리셋

### concat
axis=0 일때, sql union과 유사하게 세로방향 병합이 가능하다.
(동일한 구조의 df을 연결할때 유용하다.)

In [ ]:
member_df1 = pd.DataFrame({
    'Name': ['홍길동', '이순신'],
    'Age': [33, 48]
})
member_df2 = pd.DataFrame({
    'Name': ['신사임당', '논개'],
    'Age': [27, 55]
})

display(member_df1)
display(member_df2)

In [ ]:
# 행고정 df연결 병합

# .reset_index(drop=True): index를 0부터 다시 매기기
member_df = pd.concat([member_df1, member_df2], axis=0).reset_index(drop=True)
member_df

In [ ]:
member_df = pd.concat([member_df1, member_df2], axis=1)
member_df

In [ ]:
# 고객 주문 정보 조회
# - 아래 df을 병합해서 고객/주문/상품 정보를 하나의 df에서 조회
# - 1. customers:orders 1:n
# - 2. order:order_details 1:n
# - 3. order_details:products 1:1

# 고객 정보 데이터프레임
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4],
    'customer_name': ['Alice', 'Bob', 'Charlie', 'David']
})

# 주문 정보 데이터프레임
orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104],
    'customer_id': [1, 2, 2, 4],
    'order_date': ['2024-08-01', '2024-08-02', '2024-08-03', '2024-08-04']
})

# 제품 정보 데이터프레임
products = pd.DataFrame({
    'pid': [201, 202, 203],
    'product_name': ['Widget', 'Gadget', 'Thingamajig']
})

# 주문 내역 데이터프레임
order_details = pd.DataFrame({
    'order_id': [101, 101, 102, 103, 104],
    'product_id': [201, 203, 202, 201, 203],
    'quantity': [2, 1, 5, 2, 3]
})

In [ ]:
df = pd.merge(customers, orders, on='customer_id', how='left')
df = pd.merge(df, order_details, on='order_id', how='left')
df = pd.merge(df, products, left_on='product_id', right_on='pid', how='left')
df

## 열병합

In [ ]:
score_df = pd.DataFrame({
    "Name": ["홍길동", "신사임당", "이순신"],
    "Math": [85, 90, 78],
    "English": [88, 85, 90],
    "Science": [92, 87, 85]
})
score_df

In [ ]:
# pd.melt()
# - id_vars 식별컬럼(기준컬럼)
# - value_vars 병합하고자하는 컬럼명
# - var_name 병합된 컬럼명
# - value_name 값 컬럼명
pd.melt(
    score_df,
    id_vars=['Name'],
    value_vars=['Math', 'English', 'Science'],
    var_name='Subject',
    value_name='Score'
).sort_values('Name').reset_index(drop=True)

## 분할

In [ ]:
# qcut() 분위 분할
num_df = pd.DataFrame({
    "value": [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
})
num_df

In [ ]:
num_df['qcut_bin'] = pd.qcut(
    num_df['value'], # 분위 기준 컬럼
    q=5,  # q == 분위 수
    labels = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']  # 분위를 표시할 이름
)

num_df